# 🛒 QuickBasket - Comprehensive Grocery Data Analysis

## Business Analytics Bootcamp 2025 - Project: Decoding the Basket

**Objective**: Perform comprehensive analysis of QuickBasket grocery order dataset to optimize:
- Inventory management
- Product recommendations 
- Promotional strategies
- Online content placement

**Dataset Overview**: 
- Order metadata with timestamps and user IDs
- Order items with product details and reorder status
- Product catalog with aisle and department information
- Reference data for aisles and departments

---

## 📚 Table of Contents

1. [Data Loading & Setup](#data-loading)
2. [Exploratory Data Analysis](#eda)
3. [Customer Behavior Analysis](#customer-behavior)
4. [Product Performance Analysis](#product-analysis)
5. [Temporal Patterns Analysis](#temporal-analysis)
6. [Market Basket Analysis](#market-basket)
7. [Advanced Analytics](#advanced-analytics)
8. [Business Recommendations](#recommendations)
9. [Interactive Visualizations](#visualizations)
10. [Conclusions](#conclusions)

In [ ]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
from datetime import datetime, timedelta
import warnings
from collections import Counter
from itertools import combinations
import networkx as nx
from scipy import stats
from sklearn.preprocessing import LabelEncoder
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import os
import glob

# Configure display and plotting
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (15, 10)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("📦 All libraries imported successfully!")
print(f"🐍 Python version: {pd.__version__}")
print(f"📊 Pandas version: {pd.__version__}")
print(f"🎨 Matplotlib version: {plt.matplotlib.__version__ if hasattr(plt, 'matplotlib') else 'Available'}")

<a id="data-loading"></a>
## 1. 📂 Data Loading & Setup

In [ ]:
def load_and_combine_files(file_pattern, description):
    """Load and combine multiple CSV files matching a pattern"""
    files = glob.glob(file_pattern)
    if not files:
        print(f"⚠️ No files found for pattern: {file_pattern}")
        return pd.DataFrame()
    
    print(f"📁 Loading {len(files)} files for {description}...")
    dfs = []
    total_rows = 0
    
    for file in sorted(files):
        try:
            df = pd.read_csv(file)
            dfs.append(df)
            total_rows += len(df)
            print(f"   ✅ {file}: {len(df):,} rows")
        except Exception as e:
            print(f"   ❌ Error loading {file}: {e}")
    
    if dfs:
        combined_df = pd.concat(dfs, ignore_index=True)
        print(f"   📊 Combined {description}: {len(combined_df):,} total rows")
        return combined_df
    return pd.DataFrame()

print("🚀 Starting data loading process...")
print("=" * 60)

# Load reference data
print("\n1️⃣ Loading Reference Data")
aisles = pd.read_csv('aisle_info.csv')
departments = pd.read_csv('department_info.csv')
products = pd.read_csv('product_catalog.csv')

print(f"   ✅ Aisles: {len(aisles):,} records")
print(f"   ✅ Departments: {len(departments):,} records")
print(f"   ✅ Products: {len(products):,} records")

# Load and combine order metadata
print("\n2️⃣ Loading Order Metadata")
order_metadata = load_and_combine_files('order_metadata_*.csv', 'Order Metadata')

# Load and combine order items
print("\n3️⃣ Loading Order Items")
order_items_prior = load_and_combine_files('order_items__prior_*.csv', 'Prior Order Items')
order_items_train = load_and_combine_files('order_items__train_*.csv', 'Train Order Items')

# Combine all order items
if not order_items_prior.empty and not order_items_train.empty:
    order_items = pd.concat([order_items_prior, order_items_train], ignore_index=True)
    print(f"   📊 Combined Order Items: {len(order_items):,} total records")
elif not order_items_prior.empty:
    order_items = order_items_prior.copy()
    print(f"   📊 Order Items (prior only): {len(order_items):,} records")
elif not order_items_train.empty:
    order_items = order_items_train.copy()
    print(f"   📊 Order Items (train only): {len(order_items):,} records")
else:
    print("   ❌ No order items data found")
    order_items = pd.DataFrame()

print("\n" + "=" * 60)
print("📋 FINAL DATASET SUMMARY")
print("=" * 60)
print(f"🏪 Aisles: {len(aisles):,}")
print(f"🏢 Departments: {len(departments):,}")
print(f"📦 Products: {len(products):,}")
print(f"📋 Order Metadata: {len(order_metadata):,}")
print(f"🛒 Order Items: {len(order_items):,}")
print("=" * 60)

In [ ]:
# Data quality assessment
def assess_data_quality(df, name):
    """Comprehensive data quality assessment"""
    print(f"\n📊 DATA QUALITY ASSESSMENT: {name}")
    print("-" * 50)
    print(f"Shape: {df.shape}")
    print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Missing values
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print("\n❌ Missing Values:")
        for col, count in missing[missing > 0].items():
            print(f"   {col}: {count:,} ({count/len(df)*100:.1f}%)")
    else:
        print("\n✅ No missing values")
    
    # Duplicates
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        print(f"\n⚠️ Duplicate rows: {duplicates:,}")
    else:
        print("\n✅ No duplicate rows")
    
    # Data types
    print("\n📝 Data Types:")
    for col, dtype in df.dtypes.items():
        print(f"   {col}: {dtype}")
    
    return df

# Assess quality of all datasets
if not order_metadata.empty:
    assess_data_quality(order_metadata, "ORDER METADATA")

if not order_items.empty:
    assess_data_quality(order_items, "ORDER ITEMS")

assess_data_quality(products, "PRODUCTS")

In [ ]:
# Create comprehensive merged dataset
print("🔗 Creating comprehensive merged dataset...")

# Merge order items with products
if not order_items.empty:
    items_products = order_items.merge(products, on='product_id', how='left')
    print(f"   ✅ Merged order items with products: {len(items_products):,} records")
    
    # Add aisle information
    items_products = items_products.merge(aisles, on='aisle_id', how='left')
    print(f"   ✅ Added aisle information: {len(items_products):,} records")
    
    # Add department information
    items_products = items_products.merge(departments, on='department_id', how='left')
    print(f"   ✅ Added department information: {len(items_products):,} records")
    
    # Merge with order metadata if available
    if not order_metadata.empty:
        full_data = items_products.merge(order_metadata, on='order_id', how='left')
        print(f"   ✅ Added order metadata: {len(full_data):,} records")
    else:
        full_data = items_products
        print("   ⚠️ No order metadata available for merging")
    
    print(f"\n📊 Final merged dataset shape: {full_data.shape}")
    print(f"📊 Final merged dataset columns: {list(full_data.columns)}")
    
    # Display sample of merged data
    print("\n🔍 Sample of merged data:")
    if len(full_data) > 0:
        display_cols = ['product_name', 'aisle', 'department', 'reordered']
        available_cols = [col for col in display_cols if col in full_data.columns]
        if 'order_dow' in full_data.columns:
            available_cols.append('order_dow')
        if 'order_hour_of_day' in full_data.columns:
            available_cols.append('order_hour_of_day')
        print(full_data[available_cols].head())
else:
    print("❌ No order items data available for merging")
    full_data = pd.DataFrame()

<a id="eda"></a>
## 2. 🔍 Exploratory Data Analysis

### Task 1: Core EDA Metrics

In [ ]:
def perform_core_eda():
    """Perform core EDA as required by Task 1"""
    print("🎯 TASK 1: EXPLORATORY DATA ANALYSIS")
    print("=" * 60)
    
    results = {}
    
    # 1. Total number of orders and unique customers
    print("\n1️⃣ ORDERS & CUSTOMERS")
    if not order_metadata.empty:
        total_orders = order_metadata['order_id'].nunique()
        unique_customers = order_metadata['user_id'].nunique()
        results['total_orders'] = total_orders
        results['unique_customers'] = unique_customers
        print(f"   📋 Total Orders: {total_orders:,}")
        print(f"   👥 Unique Customers: {unique_customers:,}")
        print(f"   📊 Average Orders per Customer: {total_orders/unique_customers:.1f}")
    else:
        print("   ⚠️ Order metadata not available")
    
    # 2. Number of unique products, aisles, and departments
    print("\n2️⃣ PRODUCT CATALOG OVERVIEW")
    unique_products = products['product_id'].nunique()
    unique_aisles = aisles['aisle_id'].nunique()
    unique_departments = departments['department_id'].nunique()
    
    results.update({
        'unique_products': unique_products,
        'unique_aisles': unique_aisles,
        'unique_departments': unique_departments
    })
    
    print(f"   📦 Unique Products: {unique_products:,}")
    print(f"   🛤️ Unique Aisles: {unique_aisles:,}")
    print(f"   🏢 Unique Departments: {unique_departments:,}")
    
    # 3. Top 20 most frequently ordered products
    print("\n3️⃣ TOP 20 MOST POPULAR PRODUCTS")
    if not order_items.empty:
        product_counts = order_items['product_id'].value_counts().head(20)
        top_products = products[products['product_id'].isin(product_counts.index)].set_index('product_id')
        top_products['order_count'] = product_counts
        top_products = top_products.sort_values('order_count', ascending=False)
        
        results['top_products'] = top_products
        
        for i, (_, row) in enumerate(top_products.head(10).iterrows(), 1):
            print(f"   {i:2d}. {row['product_name'][:40]:40} ({row['order_count']:,} orders)")
        print(f"   ... (showing top 10 of 20)")
    else:
        print("   ⚠️ Order items not available")
    
    # 4. Top departments and aisles by order volume
    print("\n4️⃣ TOP DEPARTMENTS BY ORDER VOLUME")
    if not full_data.empty:
        dept_counts = full_data['department'].value_counts().head(10)
        results['top_departments'] = dept_counts
        
        for i, (dept, count) in enumerate(dept_counts.head(5).items(), 1):
            print(f"   {i}. {dept}: {count:,} items")
        
        print("\n   TOP AISLES BY ORDER VOLUME")
        aisle_counts = full_data['aisle'].value_counts().head(10)
        results['top_aisles'] = aisle_counts
        
        for i, (aisle, count) in enumerate(aisle_counts.head(5).items(), 1):
            print(f"   {i}. {aisle}: {count:,} items")
    else:
        print("   ⚠️ Full data not available for department/aisle analysis")
    
    # 5. Identify most reordered products
    print("\n5️⃣ MOST REORDERED PRODUCTS")
    if not order_items.empty:
        # Calculate reorder rate for each product
        product_reorder = order_items.groupby('product_id').agg({
            'reordered': ['count', 'sum', 'mean']
        }).round(3)
        product_reorder.columns = ['total_orders', 'reordered_times', 'reorder_rate']
        product_reorder = product_reorder[product_reorder['total_orders'] >= 50]  # Min 50 orders
        product_reorder = product_reorder.sort_values('reorder_rate', ascending=False)
        
        # Add product names
        top_reordered = product_reorder.head(10).merge(
            products[['product_id', 'product_name']], 
            left_index=True, right_on='product_id'
        )
        
        results['top_reordered'] = top_reordered
        
        for i, (_, row) in enumerate(top_reordered.iterrows(), 1):
            print(f"   {i:2d}. {row['product_name'][:35]:35} ({row['reorder_rate']:.1%} reorder rate)")
        
        avg_reorder_rate = order_items['reordered'].mean()
        print(f"\n   📊 Overall Reorder Rate: {avg_reorder_rate:.1%}")
        results['avg_reorder_rate'] = avg_reorder_rate
    else:
        print("   ⚠️ Order items not available")
    
    # 6. Average basket size
    print("\n6️⃣ BASKET SIZE ANALYSIS")
    if not order_items.empty:
        basket_sizes = order_items.groupby('order_id').size()
        avg_basket_size = basket_sizes.mean()
        median_basket_size = basket_sizes.median()
        max_basket_size = basket_sizes.max()
        
        results.update({
            'avg_basket_size': avg_basket_size,
            'median_basket_size': median_basket_size,
            'max_basket_size': max_basket_size
        })
        
        print(f"   🛒 Average Basket Size: {avg_basket_size:.1f} items")
        print(f"   📊 Median Basket Size: {median_basket_size:.1f} items")
        print(f"   📈 Largest Basket: {max_basket_size} items")
        print(f"   📊 Basket Size Distribution:")
        size_dist = basket_sizes.describe()
        for stat, value in size_dist.items():
            if stat != 'count':
                print(f"      {stat}: {value:.1f}")
    else:
        print("   ⚠️ Order items not available")
    
    return results

# Execute core EDA
eda_results = perform_core_eda()

In [ ]:
# 7. Order trends by hour, day, and frequency
def analyze_temporal_patterns():
    """Analyze temporal ordering patterns"""
    print("\n7️⃣ TEMPORAL ORDERING PATTERNS")
    print("-" * 40)
    
    if order_metadata.empty:
        print("   ⚠️ Order metadata not available for temporal analysis")
        return {}
    
    results = {}
    
    # Hour of day analysis
    print("\n   📅 HOURLY ORDER PATTERNS")
    hour_counts = order_metadata['order_hour_of_day'].value_counts().sort_index()
    peak_hour = hour_counts.idxmax()
    peak_hour_orders = hour_counts.max()
    
    results['hourly_patterns'] = hour_counts
    results['peak_hour'] = peak_hour
    
    print(f"      🕐 Peak Hour: {peak_hour}:00 ({peak_hour_orders:,} orders)")
    print(f"      🌅 Morning Rush (6-12): {hour_counts[6:13].sum():,} orders")
    print(f"      🌆 Evening Rush (15-20): {hour_counts[15:21].sum():,} orders")
    
    # Day of week analysis
    print("\n   📅 DAILY ORDER PATTERNS")
    day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    dow_counts = order_metadata['order_dow'].value_counts().sort_index()
    peak_dow = dow_counts.idxmax()
    
    results['daily_patterns'] = dow_counts
    results['peak_day'] = day_names[peak_dow]
    
    for dow, count in dow_counts.items():
        day_name = day_names[dow]
        marker = "🔥" if dow == peak_dow else "  "
        print(f"      {marker} {day_name}: {count:,} orders")
    
    # Days since prior order analysis
    print("\n   📊 REORDER FREQUENCY PATTERNS")
    if 'days_since_prior_order' in order_metadata.columns:
        days_since = order_metadata['days_since_prior_order'].dropna()
        if len(days_since) > 0:
            avg_days = days_since.mean()
            median_days = days_since.median()
            
            results['avg_days_between_orders'] = avg_days
            results['median_days_between_orders'] = median_days
            
            print(f"      📈 Average Days Between Orders: {avg_days:.1f}")
            print(f"      📊 Median Days Between Orders: {median_days:.1f}")
            
            # Frequency categories
            freq_cats = pd.cut(days_since, bins=[0, 7, 14, 30, float('inf')], 
                             labels=['Weekly (≤7)', 'Bi-weekly (8-14)', 'Monthly (15-30)', 'Infrequent (>30)'])
            freq_dist = freq_cats.value_counts()
            
            print(f"\n      📊 Customer Frequency Segments:")
            for freq, count in freq_dist.items():
                pct = count / len(days_since) * 100
                print(f"         {freq}: {count:,} customers ({pct:.1f}%)")
        else:
            print("      ⚠️ No valid days_since_prior_order data")
    
    return results

temporal_results = analyze_temporal_patterns()

<a id="customer-behavior"></a>
## 3. 👥 Customer Behavior Analysis

In [ ]:
def analyze_customer_behavior():
    """Comprehensive customer behavior analysis"""
    print("\n👥 CUSTOMER BEHAVIOR ANALYSIS")
    print("=" * 50)
    
    if order_metadata.empty:
        print("❌ Order metadata not available for customer analysis")
        return {}
    
    results = {}
    
    # Customer order frequency analysis
    print("\n1️⃣ CUSTOMER LOYALTY SEGMENTS")
    customer_orders = order_metadata.groupby('user_id')['order_id'].count()
    
    # Define loyalty segments
    def categorize_loyalty(order_count):
        if order_count == 1:
            return 'One-time (1)'
        elif order_count <= 5:
            return 'Occasional (2-5)'
        elif order_count <= 15:
            return 'Regular (6-15)'
        elif order_count <= 30:
            return 'Frequent (16-30)'
        else:
            return 'Loyal (30+)'
    
    loyalty_segments = customer_orders.apply(categorize_loyalty)
    loyalty_distribution = loyalty_segments.value_counts()
    
    results['loyalty_segments'] = loyalty_distribution
    results['customer_orders_stats'] = customer_orders.describe()
    
    print(f"   📊 Customer Loyalty Distribution:")
    for segment, count in loyalty_distribution.items():
        pct = count / len(customer_orders) * 100
        print(f"      {segment}: {count:,} customers ({pct:.1f}%)")
    
    print(f"\n   📈 Customer Order Statistics:")
    print(f"      Average orders per customer: {customer_orders.mean():.1f}")
    print(f"      Median orders per customer: {customer_orders.median():.1f}")
    print(f"      Most loyal customer: {customer_orders.max()} orders")
    
    # Customer lifetime value proxy
    if not order_items.empty:
        print("\n2️⃣ CUSTOMER VALUE ANALYSIS")
        # Calculate items per customer
        customer_items = order_items.merge(
            order_metadata[['order_id', 'user_id']], on='order_id'
        ).groupby('user_id').size()
        
        results['customer_items_stats'] = customer_items.describe()
        
        print(f"      Average items per customer: {customer_items.mean():.1f}")
        print(f"      Top 10% customers buy: {customer_items.quantile(0.9):.0f}+ items")
        print(f"      Top 1% customers buy: {customer_items.quantile(0.99):.0f}+ items")
        
        # Customer reorder behavior
        customer_reorder = order_items.merge(
            order_metadata[['order_id', 'user_id']], on='order_id'
        ).groupby('user_id')['reordered'].mean()
        
        results['customer_reorder_behavior'] = customer_reorder.describe()
        
        print(f"\n      Customer Reorder Behavior:")
        print(f"         Average reorder rate: {customer_reorder.mean():.1%}")
        print(f"         High reorder customers (>80%): {(customer_reorder > 0.8).sum():,}")
        print(f"         Low reorder customers (<20%): {(customer_reorder < 0.2).sum():,}")
    
    # Order timing behavior
    print("\n3️⃣ CUSTOMER TIMING PREFERENCES")
    customer_timing = order_metadata.groupby('user_id')['order_hour_of_day'].agg(['mean', 'std'])
    
    # Classify customers by timing consistency
    consistent_timers = (customer_timing['std'] <= 2).sum()  # Low std = consistent timing
    flexible_timers = (customer_timing['std'] > 4).sum()     # High std = flexible timing
    
    results['timing_behavior'] = {
        'consistent_timers': consistent_timers,
        'flexible_timers': flexible_timers,
        'total_customers': len(customer_timing)
    }
    
    print(f"      Consistent timers (±2 hrs): {consistent_timers:,} ({consistent_timers/len(customer_timing)*100:.1f}%)")
    print(f"      Flexible timers (±4+ hrs): {flexible_timers:,} ({flexible_timers/len(customer_timing)*100:.1f}%)")
    
    # Peak customer preferences
    customer_peak_hours = customer_timing['mean'].round().astype(int)
    peak_preferences = customer_peak_hours.value_counts().sort_index()
    
    print(f"\n      Top Customer Preferred Hours:")
    for hour, count in peak_preferences.head(5).items():
        print(f"         {hour}:00 - {count:,} customers")
    
    return results

customer_results = analyze_customer_behavior()

<a id="product-analysis"></a>
## 4. 📦 Product Performance Analysis

In [ ]:
def analyze_product_performance():
    """Comprehensive product performance analysis"""
    print("\n📦 PRODUCT PERFORMANCE ANALYSIS")
    print("=" * 50)
    
    if order_items.empty or full_data.empty:
        print("❌ Product data not available for analysis")
        return {}
    
    results = {}
    
    # Department performance
    print("\n1️⃣ DEPARTMENT PERFORMANCE")
    dept_stats = full_data.groupby('department').agg({
        'product_id': 'count',          # Total items sold
        'reordered': 'mean',            # Reorder rate
        'order_id': 'nunique'           # Unique orders
    }).round(3)
    dept_stats.columns = ['items_sold', 'reorder_rate', 'unique_orders']
    dept_stats = dept_stats.sort_values('items_sold', ascending=False)
    
    results['department_performance'] = dept_stats
    
    print(f"   📊 Top 5 Departments by Volume:")
    for i, (dept, row) in enumerate(dept_stats.head(5).iterrows(), 1):
        print(f"      {i}. {dept}:")
        print(f"         Items Sold: {row['items_sold']:,}")
        print(f"         Reorder Rate: {row['reorder_rate']:.1%}")
        print(f"         Unique Orders: {row['unique_orders']:,}")
    
    print(f"\n   🔄 Top 5 Departments by Reorder Rate:")
    top_reorder_depts = dept_stats.sort_values('reorder_rate', ascending=False)
    for i, (dept, row) in enumerate(top_reorder_depts.head(5).iterrows(), 1):
        print(f"      {i}. {dept}: {row['reorder_rate']:.1%}")
    
    # Aisle performance
    print("\n2️⃣ AISLE PERFORMANCE")
    aisle_stats = full_data.groupby('aisle').agg({
        'product_id': 'count',
        'reordered': 'mean',
        'order_id': 'nunique'
    }).round(3)
    aisle_stats.columns = ['items_sold', 'reorder_rate', 'unique_orders']
    aisle_stats = aisle_stats.sort_values('items_sold', ascending=False)
    
    results['aisle_performance'] = aisle_stats
    
    print(f"   📊 Top 5 Aisles by Volume:")
    for i, (aisle, row) in enumerate(aisle_stats.head(5).iterrows(), 1):
        print(f"      {i}. {aisle[:35]:35} ({row['items_sold']:,} items)")
    
    print(f"\n   🔄 Top 5 Aisles by Reorder Rate:")
    top_reorder_aisles = aisle_stats[aisle_stats['items_sold'] >= 100].sort_values('reorder_rate', ascending=False)
    for i, (aisle, row) in enumerate(top_reorder_aisles.head(5).iterrows(), 1):
        print(f"      {i}. {aisle[:30]:30} ({row['reorder_rate']:.1%})")
    
    # Product lifecycle analysis
    print("\n3️⃣ PRODUCT LIFECYCLE ANALYSIS")
    product_stats = full_data.groupby('product_name').agg({
        'product_id': 'count',
        'reordered': ['mean', 'sum'],
        'order_id': 'nunique'
    }).round(3)
    product_stats.columns = ['total_sold', 'reorder_rate', 'times_reordered', 'unique_orders']
    product_stats = product_stats[product_stats['total_sold'] >= 10]  # Min 10 sales
    
    # Categorize products by performance
    def categorize_product(row):
        if row['total_sold'] >= 1000 and row['reorder_rate'] >= 0.6:
            return 'Star Products'
        elif row['total_sold'] >= 1000:
            return 'High Volume'
        elif row['reorder_rate'] >= 0.7:
            return 'High Loyalty'
        elif row['total_sold'] >= 100:
            return 'Steady Sellers'
        else:
            return 'Niche Products'
    
    product_stats['category'] = product_stats.apply(categorize_product, axis=1)
    product_categories = product_stats['category'].value_counts()
    
    results['product_categories'] = product_categories
    results['product_stats'] = product_stats
    
    print(f"   📊 Product Performance Categories:")
    for category, count in product_categories.items():
        pct = count / len(product_stats) * 100
        print(f"      {category}: {count:,} products ({pct:.1f}%)")
    
    # Star products details
    if 'Star Products' in product_categories:
        star_products = product_stats[product_stats['category'] == 'Star Products'].sort_values('total_sold', ascending=False)
        print(f"\n   ⭐ Top 5 Star Products:")
        for i, (product, row) in enumerate(star_products.head(5).iterrows(), 1):
            print(f"      {i}. {product[:30]:30} ({row['total_sold']:,} sold, {row['reorder_rate']:.1%} reorder)")
    
    return results

product_results = analyze_product_performance()

<a id="temporal-analysis"></a>
## 5. ⏰ Advanced Temporal Analysis

In [ ]:
def create_temporal_visualizations():
    """Create comprehensive temporal visualizations"""
    print("\n⏰ CREATING TEMPORAL VISUALIZATIONS")
    print("=" * 50)
    
    if order_metadata.empty:
        print("❌ Order metadata not available for temporal visualizations")
        return
    
    # Create subplot layout
    fig = plt.figure(figsize=(20, 15))
    
    # 1. Hourly patterns
    plt.subplot(3, 2, 1)
    hourly_data = order_metadata['order_hour_of_day'].value_counts().sort_index()
    plt.bar(hourly_data.index, hourly_data.values, color='skyblue', alpha=0.7)
    plt.title('Orders by Hour of Day', fontsize=14, fontweight='bold')
    plt.xlabel('Hour of Day')
    plt.ylabel('Number of Orders')
    plt.xticks(range(0, 24, 2))
    plt.grid(axis='y', alpha=0.3)
    
    # Add peak hour annotation
    peak_hour = hourly_data.idxmax()
    peak_value = hourly_data.max()
    plt.annotate(f'Peak: {peak_hour}:00\n({peak_value:,} orders)', 
                xy=(peak_hour, peak_value), xytext=(peak_hour+2, peak_value*1.1),
                arrowprops=dict(arrowstyle='->', color='red'),
                fontsize=10, ha='center')
    
    # 2. Daily patterns
    plt.subplot(3, 2, 2)
    day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    daily_data = order_metadata['order_dow'].value_counts().sort_index()
    colors = ['lightcoral' if i == daily_data.idxmax() else 'lightblue' for i in daily_data.index]
    plt.bar(day_names, daily_data.values, color=colors, alpha=0.7)
    plt.title('Orders by Day of Week', fontsize=14, fontweight='bold')
    plt.xlabel('Day of Week')
    plt.ylabel('Number of Orders')
    plt.xticks(rotation=45)
    plt.grid(axis='y', alpha=0.3)
    
    # 3. Days since prior order distribution
    plt.subplot(3, 2, 3)
    if 'days_since_prior_order' in order_metadata.columns:
        days_since = order_metadata['days_since_prior_order'].dropna()
        if len(days_since) > 0:
            plt.hist(days_since[days_since <= 30], bins=30, alpha=0.7, color='lightgreen', edgecolor='black')
            plt.title('Days Since Prior Order (≤30 days)', fontsize=14, fontweight='bold')
            plt.xlabel('Days Since Prior Order')
            plt.ylabel('Frequency')
            plt.axvline(days_since.median(), color='red', linestyle='--', 
                       label=f'Median: {days_since.median():.1f} days')
            plt.legend()
            plt.grid(axis='y', alpha=0.3)
    
    # 4. Heatmap: Hour vs Day
    plt.subplot(3, 2, 4)
    if len(order_metadata) > 0:
        heatmap_data = order_metadata.groupby(['order_dow', 'order_hour_of_day']).size().unstack(fill_value=0)
        sns.heatmap(heatmap_data, cmap='YlOrRd', cbar_kws={'label': 'Orders'}, 
                   yticklabels=day_names, cbar=True)
        plt.title('Order Heatmap: Hour vs Day', fontsize=14, fontweight='bold')
        plt.xlabel('Hour of Day')
        plt.ylabel('Day of Week')
    
    # 5. Basket size over time (if order_number available)
    plt.subplot(3, 2, 5)
    if 'order_number' in order_metadata.columns and not order_items.empty:
        # Calculate basket sizes
        basket_sizes = order_items.groupby('order_id').size().reset_index()
        basket_sizes.columns = ['order_id', 'basket_size']
        
        # Merge with order metadata
        basket_evolution = basket_sizes.merge(order_metadata[['order_id', 'order_number']], on='order_id')
        
        # Group by order number and calculate mean basket size
        basket_trend = basket_evolution.groupby('order_number')['basket_size'].mean()
        
        # Plot only first 50 orders to avoid clutter
        plot_data = basket_trend.head(50)
        plt.plot(plot_data.index, plot_data.values, color='purple', linewidth=2)
        plt.title('Average Basket Size by Order Number', fontsize=14, fontweight='bold')
        plt.xlabel('Order Number')
        plt.ylabel('Average Basket Size')
        plt.grid(True, alpha=0.3)
    
    # 6. Customer ordering frequency
    plt.subplot(3, 2, 6)
    customer_order_counts = order_metadata['user_id'].value_counts()
    plt.hist(customer_order_counts[customer_order_counts <= 50], bins=25, alpha=0.7, 
             color='orange', edgecolor='black')
    plt.title('Customer Order Frequency (≤50 orders)', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Orders per Customer')
    plt.ylabel('Number of Customers')
    plt.axvline(customer_order_counts.median(), color='red', linestyle='--',
               label=f'Median: {customer_order_counts.median():.1f} orders')
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('temporal_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Temporal visualizations created and saved as 'temporal_analysis.png'")

create_temporal_visualizations()

<a id="market-basket"></a>
## 6. 🛒 Market Basket Analysis

### Task 2: Understanding Customer Buying Patterns using Product Combination Analysis

In [ ]:
def perform_market_basket_analysis():
    """Perform comprehensive market basket analysis using association rules"""
    print("\n🛒 MARKET BASKET ANALYSIS")
    print("=" * 50)
    
    if order_items.empty:
        print("❌ Order items data not available for market basket analysis")
        return {}
    
    results = {}
    
    # Prepare transaction data
    print("\n1️⃣ PREPARING TRANSACTION DATA")
    
    # Get product names for better interpretation
    items_with_names = order_items.merge(products[['product_id', 'product_name']], on='product_id', how='left')
    
    # Create transaction format: list of products per order
    transactions = items_with_names.groupby('order_id')['product_name'].apply(list).tolist()
    
    print(f"   📊 Total transactions: {len(transactions):,}")
    print(f"   📊 Average items per transaction: {np.mean([len(t) for t in transactions]):.1f}")
    print(f"   📊 Sample transaction: {transactions[0][:5] if transactions else 'No data'}...")
    
    # For computational efficiency, work with a sample if dataset is too large
    max_transactions = 10000
    if len(transactions) > max_transactions:
        print(f"   ⚠️ Using sample of {max_transactions:,} transactions for analysis")
        transactions = transactions[:max_transactions]
    
    # Convert to binary matrix format for apriori algorithm
    print("\n2️⃣ APPLYING APRIORI ALGORITHM")
    
    try:
        # Use TransactionEncoder
        te = TransactionEncoder()
        te_ary = te.fit(transactions).transform(transactions)
        df_transactions = pd.DataFrame(te_ary, columns=te.columns_)
        
        print(f"   ✅ Transaction matrix shape: {df_transactions.shape}")
        print(f"   📊 Unique products in analysis: {df_transactions.shape[1]}")
        
        # Apply Apriori algorithm - start with low support to capture patterns
        min_support = max(0.001, 50 / len(transactions))  # At least 50 occurrences or 0.1%
        print(f"   🎯 Using minimum support: {min_support:.3f}")
        
        frequent_itemsets = apriori(df_transactions, min_support=min_support, use_colnames=True, max_len=3)
        
        if len(frequent_itemsets) == 0:
            print("   ⚠️ No frequent itemsets found. Trying lower support...")
            min_support = max(0.0005, 20 / len(transactions))
            frequent_itemsets = apriori(df_transactions, min_support=min_support, use_colnames=True, max_len=3)
        
        print(f"   ✅ Found {len(frequent_itemsets)} frequent itemsets")
        results['frequent_itemsets'] = frequent_itemsets
        
        if len(frequent_itemsets) > 0:
            # Show top frequent itemsets
            print("\n   📊 Top 10 Frequent Itemsets:")
            top_itemsets = frequent_itemsets.sort_values('support', ascending=False).head(10)
            for i, (_, row) in enumerate(top_itemsets.iterrows(), 1):
                items = ', '.join(list(row['itemsets']))
                print(f"      {i:2d}. {items[:50]:50} (Support: {row['support']:.3f})")
            
            # Generate association rules
            print("\n3️⃣ GENERATING ASSOCIATION RULES")
            
            try:
                # Generate rules with different metrics
                rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.3)
                
                if len(rules) == 0:
                    print("   ⚠️ No rules found with confidence >= 0.3. Trying lower threshold...")
                    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.1)
                
                print(f"   ✅ Generated {len(rules)} association rules")
                results['association_rules'] = rules
                
                if len(rules) > 0:
                    # Sort by confidence and lift
                    rules_sorted = rules.sort_values(['confidence', 'lift'], ascending=[False, False])
                    
                    print("\n   🎯 Top 10 Association Rules (by Confidence):")
                    for i, (_, rule) in enumerate(rules_sorted.head(10).iterrows(), 1):
                        antecedent = ', '.join(list(rule['antecedents']))
                        consequent = ', '.join(list(rule['consequents']))
                        
                        print(f"\n      {i:2d}. {antecedent[:30]:30} → {consequent[:30]:30}")
                        print(f"         Support: {rule['support']:.3f} | Confidence: {rule['confidence']:.3f} | Lift: {rule['lift']:.2f}")
                    
                    # High lift rules (strong associations)
                    high_lift_rules = rules[rules['lift'] > 1.5].sort_values('lift', ascending=False)
                    
                    if len(high_lift_rules) > 0:
                        print(f"\n   🚀 High Impact Rules (Lift > 1.5): {len(high_lift_rules)} found")
                        print("\n      Top 5 High Impact Rules:")
                        for i, (_, rule) in enumerate(high_lift_rules.head(5).iterrows(), 1):
                            antecedent = ', '.join(list(rule['antecedents']))
                            consequent = ', '.join(list(rule['consequents']))
                            print(f"         {i}. {antecedent} → {consequent}")
                            print(f"            Lift: {rule['lift']:.2f}, Confidence: {rule['confidence']:.2f}")
                        
                        results['high_lift_rules'] = high_lift_rules
                else:
                    print("   ⚠️ No association rules generated")
                    
            except Exception as e:
                print(f"   ❌ Error generating association rules: {e}")
        
    except Exception as e:
        print(f"   ❌ Error in market basket analysis: {e}")
        print("   💡 Trying simplified co-occurrence analysis...")
        
        # Fallback: Simple co-occurrence analysis
        results.update(perform_cooccurrence_analysis(items_with_names))
    
    return results

def perform_cooccurrence_analysis(items_with_names):
    """Perform simplified co-occurrence analysis as fallback"""
    print("\n4️⃣ CO-OCCURRENCE ANALYSIS")
    
    # Get top products for analysis
    top_products = items_with_names['product_name'].value_counts().head(20).index.tolist()
    
    # Create co-occurrence matrix
    cooc_matrix = pd.DataFrame(0, index=top_products, columns=top_products)
    
    # Calculate co-occurrences
    order_products = items_with_names[items_with_names['product_name'].isin(top_products)]
    
    for order_id, group in order_products.groupby('order_id'):
        products_in_order = group['product_name'].unique()
        
        # Update co-occurrence counts
        for i, product1 in enumerate(products_in_order):
            for product2 in products_in_order[i+1:]:
                cooc_matrix.loc[product1, product2] += 1
                cooc_matrix.loc[product2, product1] += 1
    
    # Find strongest co-occurrences
    cooc_pairs = []
    for i in range(len(cooc_matrix)):
        for j in range(i+1, len(cooc_matrix)):
            if cooc_matrix.iloc[i, j] > 0:
                cooc_pairs.append({
                    'product1': cooc_matrix.index[i],
                    'product2': cooc_matrix.columns[j],
                    'count': cooc_matrix.iloc[i, j]
                })
    
    cooc_df = pd.DataFrame(cooc_pairs).sort_values('count', ascending=False)
    
    print(f"   📊 Top 10 Product Pairs (Co-occurrence):")
    for i, (_, row) in enumerate(cooc_df.head(10).iterrows(), 1):
        print(f"      {i:2d}. {row['product1'][:25]:25} + {row['product2'][:25]:25} ({row['count']} times)")
    
    return {'cooccurrence_matrix': cooc_matrix, 'cooccurrence_pairs': cooc_df}

# Execute market basket analysis
basket_results = perform_market_basket_analysis()

<a id="advanced-analytics"></a>
## 7. 🔬 Advanced Analytics

In [ ]:
def perform_customer_segmentation():
    """Perform advanced customer segmentation using RFM analysis"""
    print("\n🎯 CUSTOMER SEGMENTATION ANALYSIS (RFM)")
    print("=" * 50)
    
    if order_metadata.empty or order_items.empty:
        print("❌ Required data not available for customer segmentation")
        return {}
    
    # Calculate RFM metrics
    print("\n1️⃣ CALCULATING RFM METRICS")
    
    # Get latest order date for recency calculation
    latest_date = order_metadata['order_number'].max()  # Using order_number as proxy for recency
    
    # Calculate customer metrics
    customer_metrics = order_metadata.groupby('user_id').agg({
        'order_id': 'count',           # Frequency
        'order_number': ['max', 'min']  # For recency calculation
    })
    
    customer_metrics.columns = ['frequency', 'last_order', 'first_order']
    customer_metrics['recency'] = latest_date - customer_metrics['last_order']  # Days since last order (proxy)
    
    # Calculate monetary value (number of items as proxy)
    customer_items = order_items.merge(
        order_metadata[['order_id', 'user_id']], on='order_id'
    ).groupby('user_id').size()
    
    customer_metrics['monetary'] = customer_items
    customer_metrics = customer_metrics.dropna()
    
    print(f"   📊 Customer metrics calculated for {len(customer_metrics):,} customers")
    print(f"   📈 Average Frequency: {customer_metrics['frequency'].mean():.1f} orders")
    print(f"   📈 Average Monetary: {customer_metrics['monetary'].mean():.1f} items")
    
    # Score RFM components (1-5 scale)
    print("\n2️⃣ CREATING RFM SCORES")
    
    customer_metrics['R_score'] = pd.qcut(customer_metrics['recency'], 5, labels=[5,4,3,2,1])  # Lower recency = higher score
    customer_metrics['F_score'] = pd.qcut(customer_metrics['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
    customer_metrics['M_score'] = pd.qcut(customer_metrics['monetary'].rank(method='first'), 5, labels=[1,2,3,4,5])
    
    # Create RFM segments
    def segment_customers(row):
        r, f, m = int(row['R_score']), int(row['F_score']), int(row['M_score'])
        
        if r >= 4 and f >= 4 and m >= 4:
            return 'Champions'
        elif r >= 3 and f >= 3 and m >= 3:
            return 'Loyal Customers'
        elif r >= 4 and f <= 2:
            return 'New Customers'
        elif r >= 3 and f >= 3 and m <= 2:
            return 'Potential Loyalists'
        elif r <= 2 and f >= 4:
            return 'At Risk'
        elif r <= 2 and f <= 2:
            return 'Lost Customers'
        elif f >= 4 and m <= 2:
            return 'Price Sensitive'
        else:
            return 'Others'
    
    customer_metrics['Segment'] = customer_metrics.apply(segment_customers, axis=1)
    
    # Segment analysis
    segment_summary = customer_metrics.groupby('Segment').agg({
        'recency': 'mean',
        'frequency': 'mean',
        'monetary': 'mean'
    }).round(1)
    
    segment_counts = customer_metrics['Segment'].value_counts()
    
    print(f"\n   📊 Customer Segments:")
    for segment, count in segment_counts.items():
        pct = count / len(customer_metrics) * 100
        metrics = segment_summary.loc[segment]
        print(f"      {segment}: {count:,} customers ({pct:.1f}%)")
        print(f"         Avg Recency: {metrics['recency']:.1f}, Frequency: {metrics['frequency']:.1f}, Monetary: {metrics['monetary']:.1f}")
    
    return {
        'customer_metrics': customer_metrics,
        'segment_summary': segment_summary,
        'segment_counts': segment_counts
    }

def analyze_product_affinity():
    """Analyze product affinity and cross-selling opportunities"""
    print("\n🔗 PRODUCT AFFINITY ANALYSIS")
    print("=" * 40)
    
    if full_data.empty:
        print("❌ Full data not available for affinity analysis")
        return {}
    
    # Department affinity analysis
    print("\n   📊 DEPARTMENT CROSS-SELLING OPPORTUNITIES")
    
    # Find orders with multiple departments
    order_departments = full_data.groupby('order_id')['department'].nunique()
    multi_dept_orders = order_departments[order_departments > 1]
    
    print(f"      Orders with multiple departments: {len(multi_dept_orders):,} ({len(multi_dept_orders)/len(order_departments)*100:.1f}%)")
    print(f"      Average departments per multi-dept order: {multi_dept_orders.mean():.1f}")
    
    # Department co-occurrence
    dept_pairs = []
    for order_id, group in full_data.groupby('order_id'):
        departments = group['department'].unique()
        if len(departments) > 1:
            for i in range(len(departments)):
                for j in range(i+1, len(departments)):
                    dept_pairs.append((departments[i], departments[j]))
    
    dept_cooc = pd.Series(dept_pairs).value_counts().head(10)
    
    print(f"\n      Top Department Combinations:")
    for i, ((dept1, dept2), count) in enumerate(dept_cooc.items(), 1):
        print(f"         {i:2d}. {dept1} + {dept2}: {count:,} orders")
    
    return {
        'multi_dept_stats': {
            'total_multi_dept_orders': len(multi_dept_orders),
            'avg_depts_per_order': multi_dept_orders.mean()
        },
        'dept_cooccurrence': dept_cooc
    }

# Execute advanced analytics
segmentation_results = perform_customer_segmentation()
affinity_results = analyze_product_affinity()

<a id="visualizations"></a>
## 8. 📊 Comprehensive Visualizations

In [ ]:
def create_comprehensive_visualizations():
    """Create comprehensive business visualizations"""
    print("\n📊 CREATING COMPREHENSIVE VISUALIZATIONS")
    print("=" * 50)
    
    # Create a large figure with multiple subplots
    fig = plt.figure(figsize=(25, 20))
    
    # 1. Top Products Bar Chart
    if 'top_products' in eda_results:
        plt.subplot(3, 4, 1)
        top_10_products = eda_results['top_products'].head(10)
        product_names = [name[:15] + '...' if len(name) > 15 else name for name in top_10_products['product_name']]
        plt.barh(range(len(top_10_products)), top_10_products['order_count'].values, color='steelblue')
        plt.yticks(range(len(top_10_products)), product_names)
        plt.xlabel('Order Count')
        plt.title('Top 10 Most Ordered Products', fontweight='bold')
        plt.gca().invert_yaxis()
        
        # Add value labels
        for i, v in enumerate(top_10_products['order_count'].values):
            plt.text(v + max(top_10_products['order_count'].values) * 0.01, i, f'{v:,}', 
                    va='center', fontsize=9)
    
    # 2. Department Performance
    if 'top_departments' in eda_results:
        plt.subplot(3, 4, 2)
        top_depts = eda_results['top_departments'].head(8)
        plt.pie(top_depts.values, labels=[d[:12] + '...' if len(d) > 12 else d for d in top_depts.index], 
               autopct='%1.1f%%', startangle=90)
        plt.title('Department Distribution', fontweight='bold')
    
    # 3. Reorder Rate Analysis
    if not order_items.empty:
        plt.subplot(3, 4, 3)
        reorder_data = order_items['reordered'].value_counts()
        plt.bar(['First Time', 'Reordered'], reorder_data.values, 
               color=['lightcoral', 'lightgreen'])
        plt.title('First-time vs Reorder Distribution', fontweight='bold')
        plt.ylabel('Number of Items')
        
        # Add percentage labels
        total = reorder_data.sum()
        for i, v in enumerate(reorder_data.values):
            plt.text(i, v + total * 0.01, f'{v/total*100:.1f}%', ha='center', fontweight='bold')
    
    # 4. Basket Size Distribution
    if not order_items.empty:
        plt.subplot(3, 4, 4)
        basket_sizes = order_items.groupby('order_id').size()
        plt.hist(basket_sizes[basket_sizes <= 30], bins=20, alpha=0.7, color='gold', edgecolor='black')
        plt.xlabel('Basket Size')
        plt.ylabel('Frequency')
        plt.title('Basket Size Distribution (≤30 items)', fontweight='bold')
        plt.axvline(basket_sizes.mean(), color='red', linestyle='--', 
                   label=f'Mean: {basket_sizes.mean():.1f}')
        plt.legend()
    
    # 5. Customer Loyalty Segments
    if 'loyalty_segments' in customer_results:
        plt.subplot(3, 4, 5)
        loyalty_data = customer_results['loyalty_segments']
        plt.bar(range(len(loyalty_data)), loyalty_data.values, color='purple', alpha=0.7)
        plt.xticks(range(len(loyalty_data)), 
                  [label.split()[0] for label in loyalty_data.index], rotation=45)
        plt.ylabel('Number of Customers')
        plt.title('Customer Loyalty Segments', fontweight='bold')
        
        # Add count labels
        for i, v in enumerate(loyalty_data.values):
            plt.text(i, v + max(loyalty_data.values) * 0.01, f'{v:,}', 
                    ha='center', fontsize=9)
    
    # 6. Hourly Order Pattern
    if 'hourly_patterns' in temporal_results:
        plt.subplot(3, 4, 6)
        hourly_data = temporal_results['hourly_patterns']
        plt.plot(hourly_data.index, hourly_data.values, marker='o', linewidth=2, color='navy')
        plt.fill_between(hourly_data.index, hourly_data.values, alpha=0.3, color='navy')
        plt.xlabel('Hour of Day')
        plt.ylabel('Number of Orders')
        plt.title('Orders by Hour of Day', fontweight='bold')
        plt.xticks(range(0, 24, 4))
        plt.grid(True, alpha=0.3)
    
    # 7. Daily Order Pattern
    if 'daily_patterns' in temporal_results:
        plt.subplot(3, 4, 7)
        daily_data = temporal_results['daily_patterns']
        day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
        colors = ['lightcoral' if i == daily_data.idxmax() else 'skyblue' for i in daily_data.index]
        plt.bar(day_names, daily_data.values, color=colors)
        plt.xlabel('Day of Week')
        plt.ylabel('Number of Orders')
        plt.title('Orders by Day of Week', fontweight='bold')
        plt.xticks(rotation=45)
    
    # 8. Product Performance Matrix
    if 'product_categories' in product_results:
        plt.subplot(3, 4, 8)
        category_data = product_results['product_categories']
        plt.bar(range(len(category_data)), category_data.values, 
               color=plt.cm.Set3(np.linspace(0, 1, len(category_data))))
        plt.xticks(range(len(category_data)), 
                  [cat.split()[0] for cat in category_data.index], rotation=45)
        plt.ylabel('Number of Products')
        plt.title('Product Performance Categories', fontweight='bold')
    
    # 9. Department Reorder Rates
    if 'department_performance' in product_results:
        plt.subplot(3, 4, 9)
        dept_perf = product_results['department_performance'].head(8)
        plt.scatter(dept_perf['items_sold'], dept_perf['reorder_rate'], 
                   s=100, alpha=0.6, c=range(len(dept_perf)), cmap='viridis')
        
        # Add department labels
        for i, dept in enumerate(dept_perf.index):
            plt.annotate(dept[:8], (dept_perf.iloc[i]['items_sold'], dept_perf.iloc[i]['reorder_rate']),
                        xytext=(5, 5), textcoords='offset points', fontsize=8)
        
        plt.xlabel('Items Sold')
        plt.ylabel('Reorder Rate')
        plt.title('Department Performance Matrix', fontweight='bold')
        plt.grid(True, alpha=0.3)
    
    # 10. Customer Segmentation (if available)
    if 'segment_counts' in segmentation_results:
        plt.subplot(3, 4, 10)
        segment_data = segmentation_results['segment_counts']
        colors = plt.cm.Pastel1(np.linspace(0, 1, len(segment_data)))
        wedges, texts, autotexts = plt.pie(segment_data.values, labels=segment_data.index, 
                                          autopct='%1.1f%%', colors=colors, startangle=90)
        plt.title('Customer Segments (RFM)', fontweight='bold')
        
        # Make percentage text bold
        for autotext in autotexts:
            autotext.set_fontweight('bold')
    
    # 11. Top Product Pairs (if available)
    if 'cooccurrence_pairs' in basket_results:
        plt.subplot(3, 4, 11)
        cooc_data = basket_results['cooccurrence_pairs'].head(8)
        pair_labels = [f"{row['product1'][:10]}+\n{row['product2'][:10]}" 
                      for _, row in cooc_data.iterrows()]
        plt.barh(range(len(cooc_data)), cooc_data['count'].values, color='orange')
        plt.yticks(range(len(cooc_data)), pair_labels)
        plt.xlabel('Co-occurrence Count')
        plt.title('Top Product Pairs', fontweight='bold')
        plt.gca().invert_yaxis()
    
    # 12. Days Between Orders
    if 'avg_days_between_orders' in temporal_results:
        plt.subplot(3, 4, 12)
        if not order_metadata.empty and 'days_since_prior_order' in order_metadata.columns:
            days_since = order_metadata['days_since_prior_order'].dropna()
            if len(days_since) > 0:
                plt.hist(days_since[days_since <= 60], bins=30, alpha=0.7, 
                        color='lightgreen', edgecolor='black')
                plt.xlabel('Days Since Prior Order')
                plt.ylabel('Frequency')
                plt.title('Reorder Frequency Pattern', fontweight='bold')
                plt.axvline(days_since.median(), color='red', linestyle='--',
                           label=f'Median: {days_since.median():.1f} days')
                plt.legend()
    
    plt.tight_layout(pad=3.0)
    plt.savefig('comprehensive_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✅ Comprehensive visualizations created and saved as 'comprehensive_analysis.png'")

create_comprehensive_visualizations()

<a id="recommendations"></a>
## 9. 💼 Business Recommendations

### Task 3: Strategic Business Recommendations

In [ ]:
def generate_business_recommendations():
    """Generate comprehensive business recommendations based on analysis"""
    print("\n💼 STRATEGIC BUSINESS RECOMMENDATIONS")
    print("=" * 60)
    
    recommendations = {
        'inventory': [],
        'bundling': [],
        'promotions': [],
        'featured_products': [],
        'strategic': []
    }
    
    # 1. INVENTORY MANAGEMENT RECOMMENDATIONS
    print("\n1️⃣ 📦 INVENTORY MANAGEMENT RECOMMENDATIONS")
    print("-" * 45)
    
    if 'top_products' in eda_results:
        top_products = eda_results['top_products'].head(20)
        print(f"   🔥 HIGH-PRIORITY STOCK ITEMS (Top 20):")
        recommendations['inventory'].extend([
            f"Maintain high stock levels for top 20 products (representing {top_products['order_count'].sum():,} orders)",
            "Implement automated reordering for products with >500 weekly orders",
            "Set up safety stock alerts for top 5 products to prevent stockouts"
        ])
        
        for i, (_, row) in enumerate(top_products.head(5).iterrows(), 1):
            print(f"      {i}. {row['product_name']}: {row['order_count']:,} orders")
        
        print(f"\n   📈 DEMAND FORECASTING:")
        if 'top_departments' in eda_results:
            top_dept = eda_results['top_departments'].index[0]
            recommendations['inventory'].extend([
                f"Allocate 25% more warehouse space for {top_dept} department",
                "Implement demand forecasting models for seasonal products",
                "Establish supplier partnerships for high-velocity items"
            ])
            print(f"      • Focus inventory planning on {top_dept} department (highest volume)")
            print(f"      • Implement ABC analysis for category-specific inventory optimization")
    
    # 2. PRODUCT BUNDLING RECOMMENDATIONS
    print("\n2️⃣ 🎁 PRODUCT BUNDLING RECOMMENDATIONS")
    print("-" * 40)
    
    if 'cooccurrence_pairs' in basket_results:
        top_pairs = basket_results['cooccurrence_pairs'].head(10)
        print(f"   🤝 RECOMMENDED PRODUCT BUNDLES:")
        
        for i, (_, row) in enumerate(top_pairs.head(5).iterrows(), 1):
            bundle_name = f"{row['product1'][:15]} + {row['product2'][:15]}"
            print(f"      {i}. '{bundle_name}' Bundle ({row['count']} co-occurrences)")
            recommendations['bundling'].append(f"Create '{bundle_name}' bundle with 5-10% discount")
        
        recommendations['bundling'].extend([
            "Implement 'Frequently Bought Together' recommendations on product pages",
            "Create themed bundles (e.g., 'Breakfast Bundle', 'Healthy Snack Pack')",
            "Offer bundle discounts of 5-15% to increase average order value"
        ])
    
    if 'high_lift_rules' in basket_results:
        print(f"\n   🚀 HIGH-IMPACT CROSS-SELLING OPPORTUNITIES:")
        high_lift = basket_results['high_lift_rules'].head(3)
        for i, (_, rule) in enumerate(high_lift.iterrows(), 1):
            antecedent = ', '.join(list(rule['antecedents']))
            consequent = ', '.join(list(rule['consequents']))
            print(f"      {i}. When customers buy '{antecedent[:25]}', suggest '{consequent[:25]}'")
            print(f"         (Lift: {rule['lift']:.2f}, Confidence: {rule['confidence']:.1%})")
            recommendations['bundling'].append(f"Auto-suggest '{consequent}' when '{antecedent}' is in cart")
    
    # 3. PROMOTIONAL TIMING RECOMMENDATIONS
    print("\n3️⃣ ⏰ PROMOTIONAL TIMING RECOMMENDATIONS")
    print("-" * 42)
    
    if 'peak_day' in temporal_results and 'peak_hour' in temporal_results:
        peak_day = temporal_results['peak_day']
        peak_hour = temporal_results['peak_hour']
        
        print(f"   📅 OPTIMAL PROMOTION TIMING:")
        print(f"      • Peak Day: {peak_day} (highest order volume)")
        print(f"      • Peak Hour: {peak_hour}:00 (optimal notification time)")
        
        recommendations['promotions'].extend([
            f"Send promotional notifications on {peak_day} mornings for maximum reach",
            f"Schedule flash sales during {peak_hour}:00-{peak_hour+2}:00 window",
            "Launch weekend promotions on Friday evenings to capture weekend shoppers",
            "Implement time-based dynamic pricing during low-traffic hours"
        ])
        
        if 'daily_patterns' in temporal_results:
            daily_data = temporal_results['daily_patterns']
            low_day_idx = daily_data.idxmin()
            low_day = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday'][low_day_idx]
            print(f"      • Low Activity Day: {low_day} (offer special incentives)")
            recommendations['promotions'].append(f"Run '{low_day} Deals' to boost low-traffic day sales")
    
    if 'avg_days_between_orders' in temporal_results:
        avg_days = temporal_results['avg_days_between_orders']
        print(f"\n   🔄 CUSTOMER RETENTION TIMING:")
        print(f"      • Average reorder cycle: {avg_days:.1f} days")
        recommendations['promotions'].extend([
            f"Send reorder reminders after {avg_days-2:.0f} days since last order",
            "Implement personalized discount emails based on individual order patterns",
            "Create subscription services for high-frequency reorder products"
        ])
    
    # 4. FEATURED PRODUCTS RECOMMENDATIONS
    print("\n4️⃣ ⭐ FEATURED PRODUCTS & APP PLACEMENT")
    print("-" * 40)
    
    if 'product_categories' in product_results:
        print(f"   🏆 HOMEPAGE FEATURE RECOMMENDATIONS:")
        
        if 'Star Products' in product_results['product_categories']:
            print(f"      • Feature 'Star Products' prominently on homepage banner")
            recommendations['featured_products'].append("Create 'Customer Favorites' section featuring Star Products")
        
        if 'top_reordered' in eda_results:
            top_reordered = eda_results['top_reordered'].head(5)
            print(f"\n      • HIGH LOYALTY PRODUCTS (for personalized recommendations):")
            for i, (_, row) in enumerate(top_reordered.iterrows(), 1):
                print(f"         {i}. {row['product_name'][:30]:30} ({row['reorder_rate']:.1%} reorder rate)")
            
            recommendations['featured_products'].extend([
                "Create 'Reorder Favorites' quick-access section for logged-in users",
                "Implement one-click reorder functionality for high-loyalty products",
                "Feature top reordered products in personalized email campaigns"
            ])
    
    if 'aisle_performance' in product_results:
        top_aisle = product_results['aisle_performance'].index[0]
        print(f"\n   🛤️ AISLE OPTIMIZATION:")
        print(f"      • Most Popular Aisle: {top_aisle}")
        recommendations['featured_products'].extend([
            f"Create dedicated landing page for {top_aisle} category",
            "Implement aisle-based product filtering and sorting",
            "Add 'Shop by Aisle' feature to improve navigation"
        ])
    
    # 5. STRATEGIC RECOMMENDATIONS
    print("\n5️⃣ 🎯 STRATEGIC BUSINESS RECOMMENDATIONS")
    print("-" * 42)
    
    print(f"   📊 DATA-DRIVEN GROWTH STRATEGIES:")
    
    if 'loyalty_segments' in customer_results:
        loyalty_data = customer_results['loyalty_segments']
        one_time_pct = loyalty_data.get('One-time (1)', 0) / loyalty_data.sum() * 100
        
        if one_time_pct > 30:
            print(f"      🚨 HIGH ONE-TIME CUSTOMER RATE: {one_time_pct:.1f}%")
            recommendations['strategic'].extend([
                "Implement new customer onboarding sequence with personalized recommendations",
                "Offer first-time buyer incentives (10-15% discount on second order)",
                "Create welcome email series highlighting popular products and features"
            ])
    
    if 'avg_reorder_rate' in eda_results:
        reorder_rate = eda_results['avg_reorder_rate']
        print(f"\n      🔄 CUSTOMER RETENTION FOCUS:")
        print(f"         Current reorder rate: {reorder_rate:.1%}")
        
        recommendations['strategic'].extend([
            "Target 5% improvement in reorder rate through personalized experiences",
            "Implement predictive analytics to identify at-risk customers",
            "Develop loyalty program with points-based rewards system"
        ])
    
    if 'segment_counts' in segmentation_results:
        segments = segmentation_results['segment_counts']
        if 'Champions' in segments:
            champions_pct = segments['Champions'] / segments.sum() * 100
            print(f"\n      👑 CUSTOMER LIFETIME VALUE:")
            print(f"         Champion customers: {champions_pct:.1f}%")
            recommendations['strategic'].extend([
                "Create VIP program for Champion customers with exclusive benefits",
                "Implement referral program to leverage Champion customer networks",
                "Develop premium delivery options for high-value customer segments"
            ])
    
    recommendations['strategic'].extend([
        "Implement machine learning recommendation engine for personalized shopping",
        "Develop mobile app features for voice ordering and smart shopping lists",
        "Create subscription boxes for frequent purchase categories",
        "Establish data-driven supplier negotiations based on demand forecasting"
    ])
    
    # SUMMARY OF ALL RECOMMENDATIONS
    print("\n" + "=" * 60)
    print("📋 EXECUTIVE SUMMARY OF RECOMMENDATIONS")
    print("=" * 60)
    
    total_recommendations = sum(len(v) for v in recommendations.values())
    print(f"\n🎯 Total Strategic Recommendations: {total_recommendations}")
    
    print(f"\n📦 Inventory Management: {len(recommendations['inventory'])} recommendations")
    print(f"🎁 Product Bundling: {len(recommendations['bundling'])} recommendations")
    print(f"⏰ Promotional Timing: {len(recommendations['promotions'])} recommendations")
    print(f"⭐ Featured Products: {len(recommendations['featured_products'])} recommendations")
    print(f"🎯 Strategic Initiatives: {len(recommendations['strategic'])} recommendations")
    
    print("\n🚀 IMPLEMENTATION PRIORITIES:")
    priorities = [
        "1. Optimize inventory for top 20 products (immediate impact)",
        "2. Implement product bundling for top 5 co-occurrence pairs",
        "3. Launch targeted promotions during peak hours/days",
        "4. Create personalized reorder recommendations",
        "5. Develop customer retention programs for one-time buyers"
    ]
    
    for priority in priorities:
        print(f"   {priority}")
    
    return recommendations

# Generate comprehensive recommendations
business_recommendations = generate_business_recommendations()

<a id="conclusions"></a>
## 10. 🎯 Key Insights & Conclusions

In [ ]:
def generate_executive_summary():
    """Generate comprehensive executive summary"""
    print("\n🎯 EXECUTIVE SUMMARY & KEY INSIGHTS")
    print("=" * 60)
    
    # Business Impact Summary
    print("\n💼 BUSINESS IMPACT SUMMARY")
    print("-" * 30)
    
    if not order_metadata.empty and not order_items.empty:
        total_orders = len(order_metadata)
        total_customers = order_metadata['user_id'].nunique()
        total_items = len(order_items)
        avg_basket = order_items.groupby('order_id').size().mean()
        reorder_rate = order_items['reordered'].mean()
        
        print(f"📊 Business Scale:")
        print(f"   • Total Orders Analyzed: {total_orders:,}")
        print(f"   • Active Customers: {total_customers:,}")
        print(f"   • Items Sold: {total_items:,}")
        print(f"   • Average Basket Size: {avg_basket:.1f} items")
        print(f"   • Customer Reorder Rate: {reorder_rate:.1%}")
    
    # Key Findings
    print("\n🔍 KEY ANALYTICAL FINDINGS")
    print("-" * 30)
    
    findings = []
    
    # Product insights
    if 'top_products' in eda_results:
        top_product = eda_results['top_products'].iloc[0]
        findings.append(f"Most Popular Product: '{top_product['product_name']}' with {top_product['order_count']:,} orders")
    
    # Department insights
    if 'top_departments' in eda_results:
        top_dept = eda_results['top_departments'].index[0]
        dept_count = eda_results['top_departments'].iloc[0]
        findings.append(f"Leading Department: '{top_dept}' accounts for {dept_count:,} items")
    
    # Temporal insights
    if 'peak_day' in temporal_results and 'peak_hour' in temporal_results:
        findings.append(f"Peak Shopping: {temporal_results['peak_day']}s at {temporal_results['peak_hour']}:00")
    
    # Customer behavior insights
    if 'avg_days_between_orders' in temporal_results:
        avg_days = temporal_results['avg_days_between_orders']
        findings.append(f"Customer Reorder Cycle: Every {avg_days:.1f} days on average")
    
    # Market basket insights
    if 'cooccurrence_pairs' in basket_results:
        top_pair = basket_results['cooccurrence_pairs'].iloc[0]
        findings.append(f"Top Product Pairing: '{top_pair['product1']}' + '{top_pair['product2']}' ({top_pair['count']} times)")
    
    for i, finding in enumerate(findings, 1):
        print(f"   {i}. {finding}")
    
    # Strategic Opportunities
    print("\n🚀 STRATEGIC OPPORTUNITIES")
    print("-" * 30)
    
    opportunities = [
        "Revenue Growth: Implement product bundling to increase average order value by 10-15%",
        "Customer Retention: Reduce one-time customer rate through personalized onboarding",
        "Operational Efficiency: Optimize inventory based on demand forecasting models",
        "Market Expansion: Leverage peak shopping patterns for targeted marketing campaigns",
        "Customer Experience: Deploy AI-powered recommendations for personalized shopping"
    ]
    
    for i, opportunity in enumerate(opportunities, 1):
        print(f"   {i}. {opportunity}")
    
    # Success Metrics
    print("\n📈 RECOMMENDED SUCCESS METRICS")
    print("-" * 35)
    
    metrics = [
        "Average Order Value (AOV): Target 15% increase through bundling",
        "Customer Retention Rate: Improve by 10% within 6 months",
        "Reorder Rate: Increase from current level to 65%+",
        "Inventory Turnover: Optimize based on demand forecasting",
        "Customer Lifetime Value: Track improvement across segments",
        "Cross-sell Success: Monitor bundle adoption rates",
        "Personalization CTR: Measure recommendation engine effectiveness"
    ]
    
    for i, metric in enumerate(metrics, 1):
        print(f"   {i}. {metric}")
    
    # Next Steps
    print("\n🎯 RECOMMENDED NEXT STEPS")
    print("-" * 30)
    
    next_steps = [
        "Phase 1 (Week 1-2): Implement inventory optimization for top 20 products",
        "Phase 2 (Week 3-4): Launch product bundling for top 5 co-occurrence pairs",
        "Phase 3 (Month 2): Deploy targeted promotional campaigns during peak times",
        "Phase 4 (Month 3): Implement customer segmentation and personalized marketing",
        "Phase 5 (Month 4+): Deploy AI recommendation engine and advanced analytics"
    ]
    
    for step in next_steps:
        print(f"   • {step}")
    
    # ROI Estimation
    print("\n💰 ESTIMATED ROI IMPACT")
    print("-" * 25)
    
    if not order_items.empty:
        total_items = len(order_items)
        avg_basket = order_items.groupby('order_id').size().mean()
        
        print(f"   📈 Potential Business Impact:")
        print(f"      • 15% AOV increase: +{avg_basket * 0.15:.1f} items per order")
        print(f"      • 10% retention improvement: +{order_metadata['user_id'].nunique() * 0.1:.0f} retained customers")
        print(f"      • Bundle adoption (20%): {len(order_metadata) * 0.2:.0f} bundled orders")
        print(f"      • Cross-sell success: +5-10% revenue from recommendations")
    
    print("\n" + "=" * 60)
    print("🎉 ANALYSIS COMPLETED SUCCESSFULLY")
    print("=" * 60)
    print("\n📊 This comprehensive analysis provides QuickBasket with:")
    print("   ✅ Data-driven insights into customer behavior")
    print("   ✅ Actionable recommendations for business growth")
    print("   ✅ Strategic roadmap for implementation")
    print("   ✅ Measurable success metrics and ROI projections")
    
    return {
        'findings': findings,
        'opportunities': opportunities,
        'metrics': metrics,
        'next_steps': next_steps
    }

# Generate final executive summary
executive_summary = generate_executive_summary()

In [ ]:
# Export key results to CSV for stakeholder review
def export_results_to_csv():
    """Export key analysis results to CSV files"""
    print("\n📤 EXPORTING ANALYSIS RESULTS")
    print("=" * 35)
    
    export_files = []
    
    try:
        # Export top products
        if 'top_products' in eda_results:
            top_products_export = eda_results['top_products'].reset_index()
            top_products_export.to_csv('top_products_analysis.csv', index=False)
            export_files.append('top_products_analysis.csv')
        
        # Export department performance
        if 'department_performance' in product_results:
            dept_perf_export = product_results['department_performance'].reset_index()
            dept_perf_export.to_csv('department_performance.csv', index=False)
            export_files.append('department_performance.csv')
        
        # Export customer segments
        if 'segment_counts' in segmentation_results:
            segments_export = pd.DataFrame({
                'segment': segmentation_results['segment_counts'].index,
                'customer_count': segmentation_results['segment_counts'].values
            })
            segments_export.to_csv('customer_segments.csv', index=False)
            export_files.append('customer_segments.csv')
        
        # Export product pairs
        if 'cooccurrence_pairs' in basket_results:
            basket_results['cooccurrence_pairs'].head(50).to_csv('product_pairs_analysis.csv', index=False)
            export_files.append('product_pairs_analysis.csv')
        
        # Export business recommendations
        recommendations_export = []
        for category, recs in business_recommendations.items():
            for rec in recs:
                recommendations_export.append({'category': category, 'recommendation': rec})
        
        if recommendations_export:
            pd.DataFrame(recommendations_export).to_csv('business_recommendations.csv', index=False)
            export_files.append('business_recommendations.csv')
        
        print(f"   ✅ Successfully exported {len(export_files)} files:")
        for file in export_files:
            print(f"      • {file}")
            
    except Exception as e:
        print(f"   ⚠️ Export error: {e}")
    
    return export_files

exported_files = export_results_to_csv()

## 🎉 Analysis Complete!

This comprehensive notebook has analyzed the QuickBasket grocery dataset and provided:

### ✅ **Completed Analysis**
- **Exploratory Data Analysis**: Customer behavior, product performance, temporal patterns
- **Market Basket Analysis**: Product associations and cross-selling opportunities  
- **Customer Segmentation**: RFM analysis and loyalty segments
- **Advanced Analytics**: Predictive insights and optimization opportunities
- **Business Recommendations**: Strategic, data-driven action items

### 📊 **Key Deliverables**
- Comprehensive visualizations showing business insights
- Association rules for product bundling strategies
- Customer segmentation for targeted marketing
- Operational recommendations for inventory and promotions
- Exported CSV files for stakeholder review

### 🚀 **Business Impact**
- **Revenue Growth**: Product bundling and cross-selling strategies
- **Customer Retention**: Personalized recommendations and loyalty programs
- **Operational Efficiency**: Demand forecasting and inventory optimization
- **Market Intelligence**: Customer behavior insights and trend analysis

---

**Next Steps**: Implement the recommended strategies in phases, starting with inventory optimization and product bundling, then progressing to advanced personalization and predictive analytics.

*This analysis provides QuickBasket with a comprehensive roadmap for data-driven business growth and customer experience optimization.*